# 01 視覺化（講義 02a / 02b）

每個圖都先給**課程寫法**（plt），再給 statkit 的一行寫法。

In [ ]:
import math
import random
import statistics

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sympy as sp
from scipy import stats

import statkit as sk
sk.setup()  # 字體大小 16、中文字型

In [ ]:
values = [178, 172, 175, 184, 172, 175, 165, 178, 177, 175,
          180, 182, 177, 183, 180, 178, 179, 162, 170, 171]   # 講義 03a：棒球隊身高
sizes = [75, 59, 85, 65, 72, 46, 107, 91, 75, 65, 88, 59]
prices = [315, 229, 355, 261, 234, 216, 308, 306, 289, 204, 265, 195]
grades = ['A', 'A', 'B', 'C', 'B', 'A', 'D', 'B', 'C', 'B', 'A', 'F']
months = list(range(1, 13))
sales = [120, 135, 150, 145, 160, 172, 180, 176, 165, 158, 190, 210]

## 直方圖：指定組界

In [ ]:
endpoints = range(160, 196, 4)
plt.hist(values, bins=endpoints, facecolor='lightgrey', edgecolor='black')
plt.xticks(endpoints)
plt.xlabel('Heights (in cm)')
plt.ylabel('Frequency')
plt.title('棒球隊身高分布')
plt.show()

In [ ]:
sk.hist_classes(values, start=160, width=4, xlabel='Heights (in cm)', title='同上（statkit）')
plt.show()

## 直方圖 + 常態曲線（判斷是不是鐘形）

In [ ]:
sk.hist(values, bins=6, normal=True)
plt.show()

## 長條圖（類別資料）

In [ ]:
counts = pd.Series(grades).value_counts().sort_index()
plt.bar(counts.index, counts.values, facecolor='lightgrey', edgecolor='black')
plt.xlabel('Grade')
plt.ylabel('Frequency')
plt.title('成績分布')
plt.show()

## 圓餅圖 / 柏拉圖

In [ ]:
counts = pd.Series(grades).value_counts()
plt.pie(counts.values, labels=counts.index, autopct='%1.1f%%', startangle=90, counterclock=False)
plt.axis('equal')
plt.title('成績比例')
plt.show()

sk.pareto(grades, title='柏拉圖')
plt.show()

## 折線圖（時間序列）

In [ ]:
plt.plot(months, sales, marker='o', color='black')
plt.xticks(months)
plt.xlabel('Month')
plt.ylabel('Sales')
plt.title('月銷售量')
plt.show()

## 散佈圖 + 相關係數 + 迴歸線

In [ ]:
plt.figure(figsize=(6, 6))
plt.plot(sizes, prices, 'o', color='black')
plt.xlim(0, 120)
plt.ylim(0, 400)
plt.xlabel('Sizes (in squared meters)')
plt.ylabel('Prices (in $1000)')
plt.title('House sizes and prices')
plt.show()

sk.scatter_r(sizes, prices, fit=True, xlabel='Size', ylabel='Price')
plt.show()

## 盒鬚圖（多組比較）

In [ ]:
group_a = values[:10]
group_b = values[10:]
plt.boxplot([group_a, group_b], tick_labels=['A', 'B'], widths=0.3)
plt.ylabel('Height')
plt.title('兩組身高')
plt.show()

## 多張子圖、長條圖並排（講義 02b）

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].hist(values, bins=6, color='lightgrey', edgecolor='black')
axes[0].set_title('Histogram')
axes[1].boxplot(values)
axes[1].set_title('Box plot')
plt.show()

fig, ax = plt.subplots(figsize=(7, 5))
x = np.arange(3)
ax.bar(x - 0.2, [10, 15, 12], 0.4, label='2025', facecolor='lightgrey', edgecolor='black')
ax.bar(x + 0.2, [12, 14, 18], 0.4, label='2026', facecolor='dimgray', edgecolor='black')
ax.set_xticks(x)
ax.set_xticklabels(['A', 'B', 'C'])
ax.legend()
ax.set_title('並排長條圖')
plt.show()

## 多個變數一起畫（講義 02a、HW02 / HW03）

範例資料跟 `02_敘述統計.ipynb` 一樣。VS Code snippets：`barg`、`bars`、`scatg`、`scatm`、`boxg`、`linem`。

In [ ]:
rng = np.random.default_rng(1)   # 範例資料：120 戶的地區、居住地、收入、負債、食物支出（考試時換成自己的 df）
n = 120
households = pd.DataFrame({
    'Region': rng.choice([1, 2, 3, 4], n),
    'Location': rng.choice([1, 2], n),
    'Income': rng.normal(60000, 15000, n).round(),
    'Debt': rng.normal(15000, 6000, n).clip(0).round(),
})
households['Food'] = (2000 + 0.06 * households['Income'] + rng.normal(0, 800, n)).round()
households['Region'] = households['Region'].map({1: 'North East', 2: 'Middle West', 3: 'South', 4: 'West'})
households.head()

### 分組長條圖（並排 / 堆疊）

`pivot_table` 或 `crosstab` 的**列**是橫軸的類別、**欄**是不同顏色；並排時每根長條往左右位移 `width / 2`，堆疊時用 `bottom=` 指定從哪裡開始疊。

In [ ]:
table = households.pivot_table(index='Region', columns='Location', values='Food', aggfunc='mean')
x = np.arange(len(table.index))   # 每個地區的位置 0, 1, 2, 3
width = 0.35
plt.bar(x - width / 2, table[1], width, label='Metropolitan', edgecolor='black')
plt.bar(x + width / 2, table[2], width, label='Non-metropolitan', edgecolor='black')
plt.xticks(x, table.index, rotation=30, ha='right')
plt.ylabel('Average food expenditure ($)')
plt.legend()
plt.title('Average Food Expenditure by Region and Location')
plt.show()

In [ ]:
counts = pd.crosstab(households['Region'], households['Location'])
bottom = np.zeros(len(counts.index))
for col, label in [(1, 'Metropolitan'), (2, 'Non-metropolitan')]:
    plt.bar(counts.index, counts[col], bottom=bottom, label=label, edgecolor='black')
    bottom += counts[col].to_numpy()
plt.xticks(rotation=30, ha='right')
plt.ylabel('Households')
plt.legend()
plt.title('Households by Region (stacked)')
plt.show()

In [ ]:
sk.bar_grouped(table, ylabel='Average food expenditure ($)', title='同上（statkit）')
sk.bar_grouped(counts, stacked=True, ylabel='Households', title='同上（statkit，堆疊）')
plt.show()

### 依類別上色的散佈圖

每一組畫一次 `plt.scatter(..., label=組名)`，顏色會自動換，最後加 `plt.legend()`。

In [ ]:
for region, group in households.groupby('Region'):
    plt.scatter(group['Income'], group['Food'], label=region)
plt.xlabel('Income ($)')
plt.ylabel('Food expenditure ($)')
plt.legend()
plt.title('Food vs. Income by Region')
plt.show()

In [ ]:
sk.scatter_by(households, 'Income', 'Food', by='Region')   # 也會印出各組的 r
plt.show()

### 散佈圖矩陣：幾個變數兩兩之間的關係（HW03）

In [ ]:
pairs = [('Income', 'Food'), ('Debt', 'Food'), ('Income', 'Debt')]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (x, y) in zip(axes, pairs):
    r = households[x].corr(households[y])
    ax.scatter(households[x], households[y], facecolors='none', edgecolors='black')
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.set_title(f'{y} vs {x} (r = {r:.2f})')
plt.show()

In [ ]:
sk.scatter_matrix(households, ['Food', 'Income', 'Debt'])
plt.show()

### 依類別分組的盒鬚圖

In [ ]:
groups = households.groupby('Region')['Food']
plt.boxplot([g for _, g in groups], tick_labels=[name for name, _ in groups], widths=0.4)
plt.xticks(rotation=30, ha='right')
plt.ylabel('Food expenditure ($)')
plt.title('Food Expenditure by Region')
plt.show()

In [ ]:
sk.box_by(households, 'Food', by='Region')
plt.show()

### 多條折線（講義 02a：會員 vs 非會員）

每條線一個 `plt.plot(..., label=...)`，最後 `plt.legend()`。

In [ ]:
months = range(1, 13)
member = [35, 38, 52, 60, 75, 82, 90, 88, 72, 61, 45, 37]       # 範例數字
non_member = [10, 12, 20, 28, 40, 48, 55, 52, 38, 26, 15, 11]
plt.plot(months, member, marker='o', linestyle='-', color='#002676', label='member')
plt.plot(months, non_member, marker='o', linestyle='-', color='#FDB515', label='non-member')
plt.xticks(months)
plt.xlabel('Month')
plt.ylabel('Monthly rentals (in 1000)')
plt.legend()
plt.title('Monthly rentals')
plt.show()

## 莖葉圖、Q-Q 圖

In [ ]:
sk.stem_leaf(values, stem_unit=10)
sk.qq_plot(values)
plt.show()

## 小提醒
- `plt.savefig('檔名.png')` 要放在 `plt.show()` **之前**
- 圖太小：`plt.figure(figsize=(10, 6))` 或 `sk.setup(dpi=120)`
- 中文變方塊：重新執行 `sk.setup()`